# Peer Effect Linear Regression Analysis

## Objective
To investigate if a student's friends' average depression score in Wave 2 (`w2_peer_avg_mh_score`) predicts the student's own depression score in Wave 3 (`w3_mh_score`), while controlling for the student's baseline depression in Wave 2 (`w2_own_mh_score`).

## Methodology
1.  **Model 1 (Raw)**: OLS on original scores (interpret physics meaning).
2.  **Model 2 (Standardized)**: OLS on Z-scores (interpret relative importance/Beta).

---

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
from scipy.stats import zscore

# --- configuration ---
# Paths 
W2_PEER_STATS_PATH = r"../../relationship_reallike/w2_peer_mental_health_stats.csv"
W3_DATA_PATH = r"../../../../Data/2025data/TIGPS_W3_studentdata_ver4_cleaned_cols_removed_missing_common_only.csv"

# Fix absolute paths if running locally/interactively
if not os.path.exists(W2_PEER_STATS_PATH):
    W2_PEER_STATS_PATH = r"C:/Users/user/Desktop/TIGPS_Plan_data/20251229_new_progress/Code/EDA/relationship_reallike/w2_peer_mental_health_stats.csv"
    W3_DATA_PATH = r"C:/Users/user/Desktop/TIGPS_Plan_data/20251229_new_progress/Data/2025data/TIGPS_W3_studentdata_ver4_cleaned_cols_removed_missing_common_only.csv"

print(f"Loading Peer Stats from: {W2_PEER_STATS_PATH}")
print(f"Loading W3 Data from: {W3_DATA_PATH}")

In [ ]:
# 1. Load Data
peer_df = pd.read_csv(W2_PEER_STATS_PATH)
try:
    w3_df = pd.read_csv(W3_DATA_PATH, on_bad_lines='skip', engine='python')
except Exception as e:
    print(f"Error loading W3 data: {e}")

# 2. Calculate W3 Mental Health Score
mh_cols_w3 = [f"54-{i}" for i in range(1, 15)]
w3_df['w3_mh_score'] = w3_df[mh_cols_w3].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
w3_df = w3_df.dropna(subset=['w3_mh_score', 'student_id'])

# 3. Merge Datasets
merged = pd.merge(peer_df, w3_df[['student_id', 'w3_mh_score']], on='student_id', how='inner')
print(f"Merged Data (Students present in both W2 and W3): {merged.shape[0]}")

In [ ]:
# 4. Prepare Regression Data
X = merged[['w2_own_mh_score', 'w2_peer_avg_mh_score']]
X = sm.add_constant(X) # Add intercept
y = merged['w3_mh_score']

# Drop any rows with NaN
data_reg = pd.concat([X, y], axis=1).dropna()
X = data_reg[['const', 'w2_own_mh_score', 'w2_peer_avg_mh_score']]
y = data_reg['w3_mh_score']

print(f"Final samples for regression: {len(y)}")

## Model 1: Raw Regression (Original Units)
This tells us the "real world" impact. e.g., how many points does my score increase if my friend's score increases by 1 point?

In [ ]:
model_raw = sm.OLS(y, X).fit()
print(model_raw.summary())

## Model 2: Standardized Regression (Beta Coefficients)
We standardize all variables (features and target) to have Mean=0 and Std=1.
This allows us to compare the **relative importance** of variables. 
- If Beta(Own) is 0.5 and Beta(Peer) is 0.05, Own history is 10x more important than Peer influence.

In [ ]:
# Standardization (Z-score)
data_std = data_reg.copy()
cols_to_std = ['w2_own_mh_score', 'w2_peer_avg_mh_score', 'w3_mh_score']

for col in cols_to_std:
    data_std[col] = zscore(data_std[col])

X_std = data_std[['const', 'w2_own_mh_score', 'w2_peer_avg_mh_score']]
y_std = data_std['w3_mh_score']

model_std = sm.OLS(y_std, X_std).fit()
print(model_std.summary())

In [ ]:
# Compare Coefficients
beta_own = model_std.params['w2_own_mh_score']
beta_peer = model_std.params['w2_peer_avg_mh_score']

print(f"--- Standardized Beta Coefficients ---")
print(f"Beta (Own History): {beta_own:.4f}")
print(f"Beta (Peer Influence): {beta_peer:.4f}")

ratio = beta_own / beta_peer
print(f"\nInterpretation: Personal history is {ratio:.1f}x more influential than current peer influence.")